# OAI Package Inventory + Semiquant Venn
Thin notebook wrapper around `tmc_oai` production APIs.


In [6]:
from pathlib import Path
import math

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from matplotlib_venn import venn2
from IPython.display import clear_output, display

from tmc_oai import (
    build_package_inventory,
    build_semiquant_join,
    build_venn_payload,
    load_oai_env,
)


In [7]:
TIMEPOINT_MAP_CSV = None  # Optional override path; default comes from .oai_env.json
TOP_N_DEFAULT = 6


In [8]:
cfg = load_oai_env()
map_csv = Path(TIMEPOINT_MAP_CSV).expanduser().resolve() if TIMEPOINT_MAP_CSV else cfg.timepoint_map_csv

inventory_result = build_package_inventory(cfg.oai_dataset_root, map_csv)
semiquant_result = build_semiquant_join(cfg.oai_dataset_root, map_csv)

display(inventory_result.coverage_summary)
display(semiquant_result.region_summary)
print(f"Collision assets across downloaded image packages: {len(semiquant_result.collision_assets):,}")
if not semiquant_result.collision_assets.empty:
    display(semiquant_result.collision_assets.head(25))


,timepoint,meta,jpg,dicom,orphan_jpg,orphan_dicom,orphan_total
0,18MONTH,0,0,0,0,0,0
1,30MONTH,0,0,0,0,0,0
2,BASELINE,14982,372,343,0,0,0


,region,semiquant_rows,rows_with_jpeg,rows_with_dicom,rows_with_both,rows_missing_both,distinct_assets,collision_assets
0,Knee,57343,398,349,62,56658,24537,0
1,Hip,16760,226,208,32,16358,8380,0


Collision assets across downloaded image packages: 0


In [ ]:
import re
import pandas as pd

region_toggle = widgets.ToggleButtons(
    options=["Knee", "Hip"],
    value="Knee",
    description="Region",
)
column_dropdown = widgets.Dropdown(options=[], description="Category")
venn_out = widgets.Output()

_NO_CATEGORY = "__NO_CATEGORY__"
_SEMIQ_FILE_BY_REGION = {
    "Knee": "oai_kxrsemiquant01.txt",
    "Hip": "oai_hxrsemiquant01.txt",
}


def _shorten_category_label(region: str, column_name: str, raw_description: str) -> str:
    text = " ".join(str(raw_description).split())
    if not text:
        return column_name

    text = re.sub(rf"^{re.escape(region)}\s+", "", text, flags=re.IGNORECASE)

    base_part = text.split("(", 1)[0].strip()
    grade_part = ""
    location_part = ""

    oarsi_match = re.search(r"OARSI\s*grades?\s*0\s*-\s*(\d+)", text, flags=re.IGNORECASE)
    generic_grade_match = re.search(r"grades?\s*0\s*-\s*(\d+)", text, flags=re.IGNORECASE)
    if oarsi_match:
        grade_part = f"OARSI 0-{oarsi_match.group(1)}"
    elif generic_grade_match:
        grade_part = f"0-{generic_grade_match.group(1)}"

    if ")" in text:
        location_part = text.split(")", 1)[1].strip()
    elif base_part != text:
        location_part = text.replace(base_part, "", 1).strip()

    location_part = location_part.replace("compartment", "").strip()
    location_part = re.sub(r"\s+", " ", location_part)

    location_replacements = {
        "lateral": "lat",
        "medial": "med",
        "supero-lateral": "supero-lat",
        "supero-medial": "supero-med",
        "joint space narrowing": "JSN",
        "subchondral": "subchr",
        "acetabular": "acetab",
        "femoral": "femoral",
        "tibia": "tibia",
        "femur": "femur",
    }
    for source, target in location_replacements.items():
        location_part = re.sub(rf"\b{re.escape(source)}\b", target, location_part, flags=re.IGNORECASE)

    label_parts = [base_part]
    if grade_part:
        label_parts.append(grade_part)
    if location_part:
        label_parts.append(location_part)

    short = " | ".join([part.strip() for part in label_parts if part.strip()])
    return short if short else column_name


def _load_description_lookup(region: str) -> dict[str, str]:
    file_name = _SEMIQ_FILE_BY_REGION.get(region, "")
    if not file_name:
        return {}

    file_path = semiquant_result.xraymeta_package_dir / file_name
    if not file_path.exists():
        return {}

    description_row = pd.read_csv(
        file_path,
        sep="\t",
        dtype=str,
        keep_default_na=False,
        engine="python",
        nrows=1,
    )
    if description_row.empty:
        return {}

    return {
        str(column_name): str(description_row.iloc[0].get(column_name, "")).strip()
        for column_name in description_row.columns
    }


def _value_scale_key(raw_value: str) -> tuple[int, float, str]:
    value = str(raw_value).strip()
    lowered = value.lower()

    ordered_words = {
        "none": 0.0,
        "normal": 0.0,
        "mild": 1.0,
        "moderate": 2.0,
        "severe": 3.0,
        "very severe": 4.0,
    }
    for token, rank in ordered_words.items():
        if token in lowered:
            return (0, rank, lowered)

    numeric_match = re.search(r"-?\d+(?:\.\d+)?", value)
    if numeric_match:
        return (0, float(numeric_match.group(0)), lowered)

    if value == "<EMPTY>":
        return (2, 1e9, lowered)

    return (1, 1e9, lowered)


def _annotate_missing(ax, missing_count: int) -> None:
    ax.text(
        1.03,
        0.02,
        f"Missing both\n{int(missing_count):,}",
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=9,
        clip_on=False,
        bbox={"facecolor": "white", "edgecolor": "#888888", "alpha": 0.85, "boxstyle": "round,pad=0.25"},
    )

def _draw_venn_or_message(
    ax,
    *,
    only_jpg: int,
    only_dicom: int,
    both: int,
    set_colors: tuple[str, str],
    alpha: float,
    empty_message: str,
) -> None:
    only_jpg = int(only_jpg)
    only_dicom = int(only_dicom)
    both = int(both)

    if (only_jpg + only_dicom + both) == 0:
        ax.axis("off")
        ax.text(
            0.5,
            0.58,
            "No JPEG/DICOM overlap",
            transform=ax.transAxes,
            ha="center",
            va="center",
            fontsize=11,
            color="#555555",
        )
        ax.text(
            0.5,
            0.45,
            empty_message,
            transform=ax.transAxes,
            ha="center",
            va="center",
            fontsize=9,
            color="#777777",
        )
        return

    venn2(
        subsets=(only_jpg, only_dicom, both),
        set_labels=("JPEG", "DICOM"),
        set_colors=set_colors,
        alpha=alpha,
        ax=ax,
    )



_column_description_lookup = {
    region: _load_description_lookup(region)
    for region in ["Knee", "Hip"]
}

_column_display_labels = {
    region: {
        column_name: _shorten_category_label(region, column_name, _column_description_lookup.get(region, {}).get(column_name, ""))
        for column_name in semiquant_result.category_columns_by_region.get(region, [])
    }
    for region in ["Knee", "Hip"]
}


def _refresh_column_options(*_):
    region = region_toggle.value
    columns = [
        column
        for column in semiquant_result.category_columns_by_region.get(region, [])
        if "semiquant01_id" not in str(column).strip().lower()
    ]

    if columns:
        options = [
            (_column_display_labels.get(region, {}).get(column, column), column)
            for column in columns
        ]
    else:
        options = [("<no categorical columns>", _NO_CATEGORY)]

    current = column_dropdown.value
    column_dropdown.options = options
    valid_values = [value for _, value in options]
    column_dropdown.value = current if current in valid_values else options[0][1]


def _draw_venn(*_):
    with venn_out:
        clear_output(wait=True)

        region = region_toggle.value
        joined = semiquant_result.joined_by_region.get(region)
        if joined is None or joined.empty:
            print(f"No semiquant rows available for region={region}.")
            return

        selected = column_dropdown.value
        category_column = None if selected == _NO_CATEGORY else selected
        if category_column is None:
            top_n = 1
        else:
            top_n = int(
                joined[category_column]
                .astype(str)
                .str.strip()
                .replace("", "<EMPTY>")
                .nunique(dropna=False)
            )

        payload = build_venn_payload(
            joined,
            category_column=category_column,
            top_n=max(1, top_n),
        )

        value_rows = payload.by_value.to_dict("records")
        if category_column is not None:
            value_rows = sorted(value_rows, key=lambda row: _value_scale_key(str(row.get("category_value", ""))))

        n_sub = len(value_rows)
        ncols = 3
        nrows = 1 + max(1, math.ceil(n_sub / ncols))

        fig, axes = plt.subplots(nrows, ncols, figsize=(5.4 * ncols, 4.2 * nrows))
        axes = np.array(axes).reshape(-1)

        main_ax = axes[0]
        _draw_venn_or_message(
            main_ax,
            only_jpg=payload.overall.only_jpg,
            only_dicom=payload.overall.only_dicom,
            both=payload.overall.both,
            set_colors=("#4C78A8", "#F58518"),
            alpha=0.60,
            empty_message=f"Rows with JPEG/DICOM: 0 | Total rows: {payload.total_rows:,}",
        )
        title_column = (
            _column_display_labels.get(region, {}).get(category_column, category_column)
            if category_column
            else "ALL"
        )
        main_ax.set_title(
            f"{region} semiquant rows matched to downloaded images\n"
            f"Category: {title_column} | N={payload.total_rows:,}"
        )
        _annotate_missing(main_ax, payload.overall.missing_both)

        for idx, row in enumerate(value_rows, start=1):
            ax = axes[idx]
            _draw_venn_or_message(
                ax,
                only_jpg=int(row["only_jpg"]),
                only_dicom=int(row["only_dicom"]),
                both=int(row["both"]),
                set_colors=("#72B7B2", "#ECA82C"),
                alpha=0.55,
                empty_message=f"Rows with JPEG/DICOM: 0 | N={int(row['row_count']):,}",
            )
            ax.set_title(
                f"{title_column} = {row['category_value']}\n"
                f"N={int(row['row_count']):,}"
            )
            _annotate_missing(ax, int(row["missing_both"]))

        for ax in axes[(1 + n_sub):]:
            ax.axis("off")

        fig.tight_layout()
        plt.show()


_refresh_column_options()
region_toggle.observe(_refresh_column_options, names="value")
region_toggle.observe(_draw_venn, names="value")
column_dropdown.observe(_draw_venn, names="value")

display(widgets.HBox([region_toggle, column_dropdown]), venn_out)
_draw_venn()


Output()